# Lesson 3.3 — Accuracy vs Latency Trade-offs

**Level: Intermediate**

## Learning Objectives

By the end of this lesson, you will be able to:

- Quantify how model tier, retrieval, and CoT affect accuracy and latency
- Choose a configuration that meets an SLA at acceptable accuracy
- Justify a configuration decision with a trade-off table

**You will produce:** A trade-off table ranking configurations by accuracy against a latency SLA.

## Concepts

Accuracy and latency usually trade off: bigger models, retrieval, chain-of-thought, and multi-agent voting raise quality but add milliseconds-to-seconds. The architect's job is to pick the **cheapest/fastest configuration that still clears the accuracy bar within the SLA** — not the most accurate one in isolation.

Levers you control: model tier (Haiku/Sonnet/Opus), whether to retrieve, number of retrieved chunks, CoT on/off, and self-consistency/voting. We model each configuration's expected accuracy and p95 latency and filter to those that satisfy the SLA, then pick the best. This is how you 'justify configuration decisions' as the Domain 3 objective requires.

> **Reference:** [Anthropic — Reducing latency](https://docs.anthropic.com/en/docs/build-with-claude/reducing-latency)

In [ ]:
# !pip install anthropic
import os, time, json

try:
    import anthropic
    if os.environ.get("ANTHROPIC_API_KEY"):
        client = anthropic.Anthropic(); LIVE = True
    else:
        client = None; LIVE = False
except Exception as e:
    client = None; LIVE = False
    print("anthropic SDK unavailable:", e)

MODELS = {"opus":"claude-opus-4-20250514","sonnet":"claude-sonnet-4-20250514","haiku":"claude-3-5-haiku-20241022"}

def ask_claude(prompt, model=MODELS["sonnet"], system=None, max_tokens=512):
    if not LIVE:
        return f"[MOCK {model}] would answer: {prompt[:80]}..."
    msg = client.messages.create(model=model, max_tokens=max_tokens,
        system=system or "You are a helpful assistant.",
        messages=[{"role":"user","content":prompt}])
    return msg.content[0].text

print("LIVE" if LIVE else "MOCK", "mode ready")

## Code-Along: Step 1 — Model configurations with accuracy and latency

We estimate accuracy and p95 latency for candidate configurations. In production these come from your eval harness (Phase 4); here we use illustrative numbers.

In [ ]:
configs = [
  {"name":"haiku, no-RAG",        "acc":0.74, "p95_ms":350},
  {"name":"haiku, RAG(3)",        "acc":0.83, "p95_ms":700},
  {"name":"sonnet, RAG(3)",       "acc":0.90, "p95_ms":1600},
  {"name":"sonnet, RAG(5)+CoT",   "acc":0.93, "p95_ms":2600},
  {"name":"opus, RAG(5)+CoT+vote","acc":0.96, "p95_ms":6000},
]
for c in configs: print(c)

## Code-Along: Step 2 — Select the best config under an SLA

Given an accuracy floor and a latency SLA, we filter feasible configs and pick the most accurate among them — the defensible choice.

In [ ]:
def choose_config(configs, min_acc, sla_ms):
    feasible = [c for c in configs if c["acc"] >= min_acc and c["p95_ms"] <= sla_ms]
    if not feasible:
        return None, "No config meets both constraints; relax SLA or accuracy, or optimize."
    best = max(feasible, key=lambda c: c["acc"])
    return best, f"Meets acc>={min_acc} within {sla_ms}ms SLA"

print(choose_config(configs, min_acc=0.88, sla_ms=3000))
print(choose_config(configs, min_acc=0.95, sla_ms=3000))

## Exercise

Add a **cost** column (relative units) to each config and change `choose_config` to break ties / rank by *accuracy per cost* among feasible configs. Then answer: for an SLA of 2000ms and accuracy floor 0.85, which config wins, and why not the most accurate one?

In [ ]:
# TODO: add 'cost' to each config
# TODO: rank feasible configs by accuracy-per-cost
# TODO: report the winner for SLA=2000ms, min_acc=0.85 and explain

## Key Takeaways

- Pick the fastest/cheapest config that clears the accuracy bar within the SLA.
- RAG, CoT, bigger models, and voting all add latency for accuracy.
- Feasibility first (meets SLA + floor), then optimize accuracy or accuracy-per-cost.
- Ground the numbers in your eval harness, not vibes (Phase 4).

---

**Next:** [Lesson 3.4 — Observability and Monitoring at Scale](./lesson-3.4-observability-at-scale.ipynb)